# Stochastic Calculus, Step by Step · Lesson 12: Change of measure and American options

Companion notebook for the lesson. Every number in the lesson's text, tables and figures is produced by running this notebook and read from `build/outputs.json`.

> **Part 2's teaching tree.** A share at S₀ = 100 that, every four months, goes up 10% (heads, H) or down 10% (tails, T); three steps make the course's one year. The real-world chance of heads is p = 0.6. The risk-free rate is the course's r = 3% (continuously compounded), so cash grows by R = e^0.01 per step, and the strike is K = 105. Made up for teaching; every random number is simulated with a fixed seed.
>
> Educational material only, not financial advice.

1. Setup · 2. Reweighting the worlds: the Radon-Nikodym derivative · 3. Averages under the new weights · 4. The state price density · 5. The American put on the tree · 6. Supermartingales and the best time to stop · 7. Many steps · 8. Exercises · 9. Export

**Running in Google Colab?** Nothing to install: this notebook only needs numpy, pandas and matplotlib, which Colab already has. Run the cells in order.

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026

## 1. Setup

In [ ]:
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

S0, K, r, sigma, T = 100.0, 105.0, 0.03, 0.20, 1.0     # the course's numbers
N = lambda x: 0.5 * (1 + math.erf(x / math.sqrt(2)))   # standard normal distribution function
out = {"meta": {
    "series": "Stochastic Calculus, Step by Step",
    "lesson": 12,
    "quantlib_version": "not used",
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}, "inputs": {"S0": S0, "K": K, "r": r, "sigma": sigma, "T": T, "r_pct": r * 100, "sigma_pct": sigma * 100}}

In [ ]:
import itertools

S0_, u, d, p = 100.0, 1.1, 0.9, 0.6          # the teaching tree: +10% (H) or -10% (T) per step, P(H) = 0.6
steps, dt_ = 3, 1 / 3                         # three steps of four months
R = math.exp(r * dt_)                         # cash growth per step
worlds = ["".join(w) for w in itertools.product("HT", repeat=steps)]   # HHH, HHT, ..., TTT


def price(w, n):
    """The share price after the first n tosses of world w."""
    return S0_ * u ** w[:n].count("H") * d ** w[:n].count("T")


def prob(w, q=p):
    """The probability of world w (or of the first tosses of one) when heads has probability q."""
    return q ** w.count("H") * (1 - q) ** w.count("T")


out["tree"] = {"S0": S0_, "u": u, "d": d, "p": p, "steps": steps, "dt": dt_, "R": R, "K": K}
pd.DataFrame({"world": worlds, "S1": [price(w, 1) for w in worlds], "S2": [price(w, 2) for w in worlds],
              "S3": [price(w, 3) for w in worlds], "P": [prob(w) for w in worlds]}).round(4)

## 2. Reweighting the worlds: the Radon-Nikodym derivative

The real probability $P$ (heads 0.6) and the risk-neutral probability $\tilde P$ (heads $\tilde p$) give the same eight worlds different weights. Their ratio, world by world, is the **Radon-Nikodym derivative**

$$Z(\omega) = \frac{\tilde P(\omega)}{P(\omega)} .$$

It is positive and averages to 1 under $P$. Worlds with many tails get more weight under $\tilde P$ than under $P$ ($Z > 1$); worlds with many heads get less.

In [ ]:
pt = (R - d) / (u - d)
rows = [{"world": w, "P": prob(w), "Pt": prob(w, pt), "Z": prob(w, pt) / prob(w), "S3": price(w, 3)} for w in worlds]
EZ = sum(x["P"] * x["Z"] for x in rows)
assert abs(EZ - 1) < 1e-12
out["rn"] = {"pt": pt, "rows": rows, "EZ": EZ, "z_up": pt / p, "z_down": (1 - pt) / (1 - p),
             "Z_HHH": rows[0]["Z"], "Z_TTT": rows[-1]["Z"]}
pd.DataFrame(rows).round(4)

## 3. Averages under the new weights

Any $\tilde P$-average is a $P$-average with $Z$ as an extra weight: $\tilde E[X] = E[ZX]$. Seen from time $n$, the weight so far is $Z_n = E[Z \mid \mathcal F_n] = (\tilde p/p)^{\#H}\big((1 - \tilde p)/(1 - p)\big)^{\#T}$, counted over the first $n$ tosses, and $Z_n$ is a $P$-martingale.

In [ ]:
E = lambda f, q=p: sum(prob(w, q) * f(w) for w in worlds)
Zw = {x["world"]: x["Z"] for x in rows}
call = lambda w: max(price(w, 3) - K, 0.0)
checks = {"ES3_rn": E(lambda w: price(w, 3), pt), "EZS3": E(lambda w: Zw[w] * price(w, 3)),
          "Ecall_rn": E(call, pt), "EZcall": E(lambda w: Zw[w] * call(w)), "ES3_real": E(lambda w: price(w, 3))}
Zn = lambda w, n: (pt / p) ** w[:n].count("H") * ((1 - pt) / (1 - p)) ** w[:n].count("T")
mart_gap = max(abs(p * Zn(w[:n] + "H", n + 1) + (1 - p) * Zn(w[:n] + "T", n + 1) - Zn(w, n)) for w in worlds for n in range(steps))
checks["Zn_gap"] = mart_gap
checks["Zn"] = {f"n{n}": {"H" * n: Zn("H" * n, n), "T" * n: Zn("T" * n, n)} for n in range(steps + 1)}
assert abs(checks["ES3_rn"] - checks["EZS3"]) < 1e-9 and mart_gap < 1e-12
out["avg"] = checks
checks

## 4. The state price density

Combine the reweighting with discounting: $\zeta = Z/R^N$. Then the price of any payoff is a **real-world** average,

$$V_0 = \frac{\tilde E[V_N]}{R^N} = E[\zeta\,V_N] .$$

$\zeta(\omega)$ is the price today of one dollar paid only in world $\omega$, per unit of real probability. It is largest in the worst worlds: a dollar there is scarce, and valuable.

In [ ]:
for x in rows:
    x["zeta"] = x["Z"] / R**steps
    x["state_price"] = x["Pt"] / R**steps
out["spd"] = {"call": E(lambda w: Zw[w] / R**steps * call(w)), "call_rn": E(call, pt) / R**steps,
              "zeta_HHH": rows[0]["zeta"], "zeta_TTT": rows[-1]["zeta"], "sum_state_prices": sum(x["state_price"] for x in rows),
              "bond": R ** -steps}
out["spd"]

## 5. The American put on the tree

An **American** option can be exercised at any step, not just at the end. Working backward, at each node its holder compares exercising now (the **intrinsic value** $\max(105 - S_n, 0)$) with holding on (the discounted risk-neutral average of the next step, the **continuation value**), and takes the larger:

$$V_n = \max\Big(105 - S_n,\ \frac{\tilde pV^{u}_{n+1} + (1 - \tilde p)V^{d}_{n+1}}{R}\Big) .$$

In [ ]:
S = lambda n, k_: S0_ * u**k_ * d ** (n - k_)
intrinsic = lambda s: max(K - s, 0.0)
Va = {(steps, k_): intrinsic(S(steps, k_)) for k_ in range(steps + 1)}
Ve = dict(Va)
info = {}
for n in range(steps - 1, -1, -1):
    for k_ in range(n + 1):
        cont = (pt * Va[(n + 1, k_ + 1)] + (1 - pt) * Va[(n + 1, k_)]) / R
        Va[(n, k_)] = max(intrinsic(S(n, k_)), cont)
        Ve[(n, k_)] = (pt * Ve[(n + 1, k_ + 1)] + (1 - pt) * Ve[(n + 1, k_)]) / R
        info[(n, k_)] = {"cont": cont, "exercise": intrinsic(S(n, k_)) > cont}
nodes = [{"t": n, "k": k_, "price": S(n, k_), "value": Va[(n, k_)], "european": Ve[(n, k_)],
          "intrinsic": intrinsic(S(n, k_)), "cont": info.get((n, k_), {}).get("cont"),
          "highlight": info.get((n, k_), {}).get("exercise", False)} for n in range(steps + 1) for k_ in range(n + 1)]
out["amer"] = {"nodes": nodes, "V0": Va[(0, 0)], "euro": Ve[(0, 0)], "premium": Va[(0, 0)] - Ve[(0, 0)],
               "exercise_nodes": [{"t": n, "k": k_, "S": S(n, k_)} for (n, k_), v in info.items() if v["exercise"]]}
pd.DataFrame(nodes).round(4)

## 6. Supermartingales and the best time to stop

Because each node takes the larger of two numbers, the discounted American value can only drift down on average: it is a **supermartingale** under $\tilde p$. It falls exactly at the nodes where the holder should already have exercised. The best rule is to stop the first time the value equals the intrinsic value; up to that time the discounted value is a martingale, and the average discounted payoff of the rule equals the price. Other rules do worse:

In [ ]:
def stop_value(rule):
    # The risk-neutral average of the discounted payoff when exercising at the first step where rule(n, k) holds.
    total = 0.0
    for w in worlds:
        k_ = 0
        for n in range(steps + 1):
            if n > 0:
                k_ += w[n - 1] == "H"
            if rule(n, k_) or n == steps:
                total += prob(w, pt) * intrinsic(S(n, k_)) / R**n
                break
    return total


best = lambda n, k_: n < steps and info.get((n, k_), {}).get("exercise", False) or (n == steps)
rules = {"best (value = intrinsic)": stop_value(best), "at expiry only": stop_value(lambda n, k_: False),
         "as soon as in the money": stop_value(lambda n, k_: n > 0 and intrinsic(S(n, k_)) > 0),
         "immediately": stop_value(lambda n, k_: True)}
super_gap = [((pt * Va[(n + 1, k_ + 1)] + (1 - pt) * Va[(n + 1, k_)]) / R - Va[(n, k_)])
             for n in range(steps) for k_ in range(n + 1)]
out["stop"] = {"rules": rules, "max_drift": max(super_gap), "min_drift": min(super_gap),
               "best": rules["best (value = intrinsic)"], "expiry": rules["at expiry only"],
               "greedy": rules["as soon as in the money"], "now": rules["immediately"]}
assert abs(rules["best (value = intrinsic)"] - Va[(0, 0)]) < 1e-12
out["stop"]["table"] = [{"rule": k_, "value": v} for k_, v in rules.items()]
out["stop"]

## 7. Many steps

With the course's numbers on a CRR tree ($u = e^{\sigma\sqrt{\Delta t}}$), the American put with strike 105 settles as $N$ grows, and stays above the European put: the right to exercise early is worth something for a put.

In [ ]:
def crr_put(N, american):
    dt = T / N
    uu = math.exp(sigma * math.sqrt(dt)); dd = 1 / uu; RR = math.exp(r * dt)
    q = (RR - dd) / (uu - dd)
    s = S0 * uu ** np.arange(N, -1, -1) * dd ** np.arange(0, N + 1)
    v = np.maximum(K - s, 0.0)
    for n in range(N - 1, -1, -1):
        s = s[:-1] / uu
        v = (q * v[:-1] + (1 - q) * v[1:]) / RR
        if american:
            v = np.maximum(v, K - s)
    return float(v[0])


many = [{"N": n, "american": crr_put(n, True), "european": crr_put(n, False)} for n in (10, 50, 100, 500, 1000)]
for x in many:
    x["premium"] = x["american"] - x["european"]
d1 = (math.log(S0 / K) + (r + 0.5 * sigma**2) * T) / (sigma * math.sqrt(T)); d2 = d1 - sigma * math.sqrt(T)
out["many"] = {"rows": many, "bs_put": K * math.exp(-r * T) * N(-d2) - S0 * N(-d1)}
pd.DataFrame(many)

## 8. Exercises

Try each one before looking anything up. Full solutions are in the companion volume, *Part 2: Solutions*.

**Exercise 1.** Compute $Z = \tilde P/P$ for the eight worlds by hand (write it as $(\tilde p/p)^{\#\text{H}}((1 - \tilde p)/(1 - p))^{\#\text{T}}$) and check that $E[Z] = 1$.

**Exercise 2.** Prove that $\tilde E[X] = E[ZX]$ for any random variable $X$ on the tree, and check it for $X = S_3$.

**Exercise 3.** Show that $Z_n = E[Z \mid \mathcal F_n]$ equals $(\tilde p/p)^{\#\text{H}}((1 - \tilde p)/(1 - p))^{\#\text{T}}$ over the first $n$ tosses, and that $Z_n$ is a martingale under $P$.

**Exercise 4.** Two probabilities are **equivalent** if they agree on which events have probability 0. Why must $\tilde P$ be equivalent to $P$ for pricing to make sense? What goes wrong if the real probability of heads is $p = 1$?

**Exercise 5.** Show that an American call on this share (no dividends) is never exercised early: at every node its continuation value is at least $S_n - 105/R^{3 - n} > S_n - 105$. So its price equals the European call's.

**Exercise 6.** Price the American put with strike 105 that expires after **two** steps by hand. At which node is it exercised early?

**Exercise 7.** (Code) With the course's numbers, compute the American and European puts for $N = 100, 200, \dots, 1000$, and find, at the step closest to six months, the highest share price at which exercising is best (the **exercise boundary**).

**Exercise 8.** (Code) Simulate 200,000 paths of the three-step teaching tree under $\tilde p$ and estimate the average discounted payoff of the best stopping rule and of *exercise as soon as in the money*. Compare with the exact values.

## 9. Export

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")